# Local ESCI recommender training workbench

Run model training deliberately from this notebook. The Streamlit apps only serve the active model artifacts and collect searches and feedback.

The first run trains from the Kaggle ESCI training partition. Later runs include feedback saved in SQLite after the dashboard reports that the 1,000-label threshold is reached. Set `FORCE_TRAIN = True` to intentionally retrain sooner.

The trainer publishes a versioned artifact folder and updates `current.json` only after training finishes. The app loads the XGBoost ranker from its native JSON file and the two-tower model from its Keras file; the product tower embeddings are saved as NumPy arrays. Python pickle is not needed.

In [ ]:
from pathlib import Path
import os

# Support opening this notebook either from the repository root or its recommender_lab folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "recommender_lab").is_dir() and (PROJECT_ROOT.parent / "recommender_lab").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "recommender_lab").is_dir():
    raise RuntimeError("Open the project repository as your VS Code/Jupyter workspace first.")
os.chdir(PROJECT_ROOT)

# Edit these two paths to point to your downloaded Kaggle ESCI data files.
EXAMPLES_PATH = Path(r"C:\data\shopping_queries_dataset_examples.parquet")
PRODUCTS_PATH = Path(r"C:\data\shopping_queries_dataset_products.parquet")

os.environ["ESCI_EXAMPLES_PATH"] = str(EXAMPLES_PATH)
os.environ["ESCI_PRODUCTS_PATH"] = str(PRODUCTS_PATH)
os.environ.setdefault("RECOMMENDER_DB_PATH", "recommender_lab/data/interactions.sqlite3")
os.environ.setdefault("RECOMMENDER_ARTIFACT_DIR", "recommender_lab/artifacts")
os.environ.setdefault("RETRAIN_THRESHOLD", "1000")

for data_path in (EXAMPLES_PATH, PRODUCTS_PATH):
    if not data_path.is_file():
        raise FileNotFoundError(f"Update the data path; file not found: {data_path}")

ARTIFACT_DIR = Path(os.environ["RECOMMENDER_ARTIFACT_DIR"])
RETRAIN_THRESHOLD = int(os.environ["RETRAIN_THRESHOLD"])
FORCE_TRAIN = False  # Change to True only when you intentionally want to override the threshold.
print(f"Project: {PROJECT_ROOT}")
print(f"Artifacts: {ARTIFACT_DIR}")
print(f"Retraining threshold: {RETRAIN_THRESHOLD} new labeled query-product pairs")

## Check the data files

This previews only a few rows so the full datasets are not loaded into notebook memory just for inspection. The trainer loads the files when you start a training run.

In [ ]:
import pandas as pd

def preview_table(path, rows=5):
    if path.suffix.lower() in {".parquet", ".pq"}:
        import pyarrow.parquet as pq
        batch = next(pq.ParquetFile(path).iter_batches(batch_size=rows))
        return batch.to_pandas()
    return pd.read_csv(path, nrows=rows)

display(preview_table(EXAMPLES_PATH))
display(preview_table(PRODUCTS_PATH))

## Check the feedback batch

Feedback comes from the customer simulator or any human labels recorded in the same SQLite database. The trainer uses one label per query-product pair and prefers human labels when both sources exist.

In [ ]:
from recommender_lab.storage import (
    feedback_count,
    init_db,
    last_trained_feedback_count,
    load_training_feedback,
    new_feedback_count,
)

init_db()
TOTAL_LABELS = feedback_count()
LAST_TRAINED_LABELS = last_trained_feedback_count()
PENDING_LABELS = new_feedback_count()
ACTIVE_POINTER = ARTIFACT_DIR / "current.json"
HAS_ACTIVE_MODEL = ACTIVE_POINTER.is_file()

print(f"Unique labeled query-product pairs: {TOTAL_LABELS}")
print(f"Already included in the last successful training run: {LAST_TRAINED_LABELS}")
print(f"New labels available: {PENDING_LABELS} / {RETRAIN_THRESHOLD}")
print(f"Active model exists: {HAS_ACTIVE_MODEL}")

## Train and publish model artifacts

`train_from_files` contains the reusable training implementation. This notebook controls when it runs and passes in the latest saved feedback. XGBoost and the two-tower model are trained together.

In [ ]:
from recommender_lab.storage import mark_training_finished, mark_training_started
from recommender_lab.train import train_from_files

SHOULD_TRAIN = (
    not HAS_ACTIVE_MODEL
    or PENDING_LABELS >= RETRAIN_THRESHOLD
    or FORCE_TRAIN
)

if not SHOULD_TRAIN:
    print("No training run started. Run this cell again after the dashboard reaches the threshold, or set FORCE_TRAIN = True.")
else:
    FEEDBACK_FOR_TRAINING = load_training_feedback()
    RUN_ID = mark_training_started(PENDING_LABELS)
    try:
        MODEL_VERSION = train_from_files(
            str(EXAMPLES_PATH),
            str(PRODUCTS_PATH),
            feedback_df=FEEDBACK_FOR_TRAINING,
            artifact_root=ARTIFACT_DIR,
        )
    except Exception as exc:
        mark_training_finished(RUN_ID, "failed", message=str(exc))
        raise
    else:
        mark_training_finished(
            RUN_ID,
            "succeeded",
            model_version=MODEL_VERSION,
            message=f"Trained with {len(FEEDBACK_FOR_TRAINING)} feedback rows after de-duplication.",
            trained_through_feedback_count=TOTAL_LABELS,
        )
        print(f"Published model version: {MODEL_VERSION}")
        print("In the customer app, click 'Reload model artifacts' to load this version.")

## What happens next

The Streamlit apps do not train or load Python pickle files. They read the version pointer and load the saved native artifacts. Keep this notebook as the controlled training step; use the dashboard to monitor new labels and decide when to run it.